# 04 - Produits et marques controversés

**Objectif 2** : repérer les produits dont les indicateurs se contredisent ou appellent la vigilance, et les marques où ces situations se concentrent.

**Les cinq signaux**

| Signal | Définition |
|---|---|
| Bien noté mais ultra-transformé | Nutri-Score A ou B et NOVA 4 |
| Bien noté mais beaucoup d'additifs | Nutri-Score A ou B et au moins 5 additifs |
| Additif à risque élevé de surexposition | Au moins un additif pour lequel l'EFSA estime qu'une partie de la population risque de dépasser la dose journalière admissible |
| Rappel de produit | Code-barres présent dans RappelConso |
| Bio mais ultra-transformé | Label bio et NOVA 4 |

**Score de controverse** : nombre de signaux (0 à 5). Chaque point correspond à un fait vérifiable sur la fiche du produit, et chaque produit signalé est accompagné d'une phrase d'explication.

**Ce que ce score n'est pas** : un jugement de dangerosité. Un risque de surexposition concerne une consommation répétée à l'échelle de la population, pas un produit isolé. Un rappel peut tenir à un défaut ponctuel de fabrication.

Prérequis : `uv run python -m nutrilys.analysis.controversies`.

In [25]:
import pandas as pd

from nutrilys import config
from nutrilys.analysis import controversies as ctv
from nutrilys.analysis.bio import subgroup_label
from nutrilys.analysis.nutrition import label_food_groups
from nutrilys.db import products_connection
from nutrilys.viz import charts
from nutrilys.viz.fmt import fmt_int, fmt_pct
from nutrilys.viz.theme import NOVA_COLORS, NOVA_LABELS, PRIMARY, save_figure, use_theme

use_theme()
con = products_connection()
signals = pd.read_parquet(config.CONTROVERSIES_PATH)
additives = pd.read_parquet(config.ADDITIVES_PATH)
n_products = con.execute("SELECT count(*) FROM products").fetchone()[0]
findings = []
print(f"{fmt_int(len(signals))} produits signalés sur {fmt_int(n_products)}")

102 012 produits signalés sur 1 171 729


## 1. Vue d'ensemble

In [26]:
counts = ctv.signal_counts(signals)
counts["text"] = counts["products"].map(fmt_int)
top = counts.iloc[0]

fig = charts.ranked_bars(
    counts,
    "label",
    "products",
    text="text",
    title=f"Signal le plus fréquent : « {top['label'].lower()} », "
    f"{fmt_int(top['products'])} produits",
    subtitle=f"Nombre de produits concernés par chaque signal, sur {fmt_int(n_products)} produits",
    value_label="Produits",
)
fig.update_xaxes(tickformat=",d")
save_figure(fig, "controversy_signals")
fig.show()

In [27]:
dist = (
    signals["controversy_score"]
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .reset_index(name="products")
)
dist["share"] = 100 * dist["products"] / dist["products"].sum()
multi = dist.loc[dist["score"] >= 2, "products"].sum()

fig = charts.category_bars(
    dist,
    "score",
    "share",
    {s: PRIMARY for s in dist["score"]},
    counts="products",
    labels={s: f"{s} signal" + ("s" if s > 1 else "") for s in dist["score"]},
    title=f"{fmt_int(multi)} produits cumulent au moins deux signaux",
    subtitle="Répartition des produits signalés selon leur score de controverse",
    value_label="Part des produits signalés (%)",
)
save_figure(fig, "controversy_score_distribution")
findings.append(
    f"{fmt_int(len(signals))} produits signalés, dont {fmt_int(multi)} avec au moins deux signaux."
)
fig.show()

## 2. Bien notés mais ultra-transformés

C'est le paradoxe le plus fréquent : un bon Nutri-Score ne dit rien du degré de transformation. Dans quels rayons le rencontre-t-on ?

In [28]:
by_sub = ctv.good_score_ultra_by_subgroup(con)
by_sub["label"] = by_sub["food_subgroup"].map(subgroup_label)
top_sub = by_sub.head(15)
lead = top_sub.iloc[0]

fig = charts.ranked_bars(
    top_sub,
    "label",
    "share",
    suffix=" %",
    value_format=".0f",
    title=f"{lead['label']} : {fmt_pct(lead['share'])} des produits notés A ou B "
    "sont ultra-transformés",
    subtitle="Part d'ultra-transformés (NOVA 4) parmi les produits notés A ou B, "
    "15 sous-groupes les plus concernés (au moins 200 produits notés A ou B)",
    value_label="Part de NOVA 4 parmi les produits A ou B (%)",
)
save_figure(fig, "good_score_ultra_by_subgroup")
findings.append(
    f"Bien noté mais ultra-transformé : surtout dans le sous-groupe "
    f"{lead['label'].lower()} ({fmt_pct(lead['share'])} des produits A ou B)."
)
fig.show()

## 3. Additifs à risque de surexposition

Évaluations de l'EFSA reprises par Open Food Facts. Population : produits avec liste d'ingrédients.

In [29]:
freq = ctv.risky_additives_frequency(con, additives)
freq["label"] = freq["name_fr"] + " (" + freq["risk_label"].str.lower() + ")"
first = freq.iloc[0]

fig = charts.ranked_bars(
    freq,
    "label",
    "share",
    suffix=" %",
    value_format=".1f",
    title=f"{first['name_fr']} est l'additif à risque le plus répandu, "
    f"présent dans {fmt_pct(first['share'], 1)} des produits",
    subtitle="Additifs à risque de surexposition élevé ou modéré selon l'EFSA, "
    "par part des produits qui en contiennent",
    value_label="Part des produits contenant l'additif (%)",
)
save_figure(fig, "risky_additives")
fig.show()

In [30]:
risky_nova = ctv.risky_share_by_nova(con, signals)
r4 = risky_nova.set_index("nova_group").loc[4, "share"]

fig = charts.category_bars(
    risky_nova,
    "nova_group",
    "share",
    NOVA_COLORS,
    labels=NOVA_LABELS,
    counts="products",
    title=f"{fmt_pct(r4)} des produits ultra-transformés contiennent un additif "
    "à risque élevé de surexposition",
    subtitle="Part des produits contenant au moins un additif à risque élevé, par groupe NOVA",
)
save_figure(fig, "risky_additives_by_nova")
findings.append(f"Additifs à risque élevé : présents dans {fmt_pct(r4)} des produits NOVA 4.")
fig.show()

## 4. Rappels de produits

Rapprochement par code-barres entre RappelConso et Open Food Facts. Les rappels portent surtout sur la sécurité sanitaire (contamination, corps étranger, allergène non déclaré), une dimension distincte de la qualité nutritionnelle.

In [31]:
recalled = signals[signals["recalled"]]
findings.append(
    f"Rappels : {fmt_int(len(recalled))} produits d'Open Food Facts retrouvés dans RappelConso."
)
print(f"{fmt_int(len(recalled))} produits rappelés retrouvés dans la table de référence")

by_group = (
    label_food_groups(recalled.assign(food_group=recalled["food_group"].fillna("non renseigné")))
    .groupby("food_group_label")
    .size()
    .sort_values(ascending=False)
    .rename("Produits rappelés")
)
by_group.to_frame()

6 064 produits rappelés retrouvés dans la table de référence


,Produits rappelés
food_group_label,
non renseigné,1702
Produits laitiers,1354
"Viandes, poissons, œufs",1172
Plats composés,422
Céréales et pommes de terre,394
Produits sucrés,312
Fruits et légumes,229
Snacks salés,210
Matières grasses et sauces,128


In [32]:
reasons = (
    recalled["recall_reason"]
    .dropna()
    .str.strip()
    .str[:90]
    .value_counts()
    .head(10)
    .rename_axis("Motif du rappel")
    .reset_index(name="Produits")
)
reasons

,Motif du rappel,Produits
0,teneur supérieure à la limite autorisée en oxy...,186
1,rupture de la chaine du froid,157
2,présence de listeria monocytogenes,124
3,teneur en oxyde d’éthylène supérieure à la lim...,103
4,suite à une alerte européenne concernant la pr...,71
5,teneur supérieure à la limite autorisée en oxy...,62
6,alerte du fournisseur de mélange d'additifs st...,54
7,détection listeria dans l'environnement de fab...,44
8,utilisation d'une matière première présentant ...,43
9,la société froneri procède au rappel de certai...,43


## 5. Marques : le « halo » du bon Nutri-Score

Pour chaque marque, on regarde ses produits notés A ou B et la part d'entre eux qui sont ultra-transformés. Un halo élevé signifie que la bonne note de la marque repose souvent sur des produits ultra-transformés.

Population : marques ayant au moins 30 produits avec Nutri-Score et NOVA, dont au moins 10 notés A ou B.

In [33]:
brands = ctv.brand_profiles(con)
halo = brands.dropna(subset=["halo"]).sort_values("halo", ascending=False).head(15)
halo["label"] = halo["brand"] + " (" + halo["n_ab"].map(fmt_int) + ")"
lead_brand = halo.iloc[0]

fig = charts.ranked_bars(
    halo,
    "label",
    "halo",
    suffix=" %",
    value_format=".0f",
    title=f"{lead_brand['brand']} : {fmt_pct(lead_brand['halo'])} de ses produits notés A ou B "
    "sont ultra-transformés",
    subtitle=f"Les 15 marques au halo le plus élevé, sur {fmt_int(brands['halo'].notna().sum())} "
    "marques éligibles (entre parenthèses : nombre de produits A ou B)",
    value_label="Part de NOVA 4 parmi les produits notés A ou B (%)",
)
save_figure(fig, "brand_halo")
findings.append(
    f"Marques : {fmt_int((brands['halo'] >= 50).sum())} marques ont plus de la "
    "moitié de leurs produits A ou B ultra-transformés."
)
fig.show()

## 6. Marques bio et ultra-transformation

Marques dont au moins 80 % des produits notés portent un label bio, classées par part d'ultra-transformés.

In [34]:
organic_brands = brands[brands["share_organic"] >= 80].sort_values("share_nova4", ascending=False)
top_org = organic_brands.head(15).copy()
top_org["label"] = top_org["brand"] + " (" + top_org["products"].map(fmt_int) + ")"

if top_org.empty:
    print("Aucune marque bio n'atteint le seuil de 30 produits notés.")
else:
    lead_org = top_org.iloc[0]
    fig = charts.ranked_bars(
        top_org,
        "label",
        "share_nova4",
        suffix=" %",
        value_format=".0f",
        title=f"Marque bio la plus transformée : {lead_org['brand']}, "
        f"{fmt_pct(lead_org['share_nova4'])} de produits NOVA 4",
        subtitle=f"{fmt_int(len(organic_brands))} marques bio éligibles, "
        "15 premières (entre parenthèses : produits notés)",
        value_label="Part de produits ultra-transformés (%)",
    )
    save_figure(fig, "organic_brands_nova4")
    findings.append(
        f"Bio : {fmt_int(signals['organic_ultra'].sum())} produits bio "
        "ultra-transformés ; parmi les marques bio, jusqu'à "
        f"{fmt_pct(lead_org['share_nova4'])} de NOVA 4 ({lead_org['brand']})."
    )
    fig.show()

## 7. Exemples : les produits populaires les plus controversés

Produits cumulant au moins deux signaux, classés par nombre de scans (popularité sur Open Food Facts).

In [35]:
examples = (
    signals[signals["controversy_score"] >= 2]
    .sort_values(["controversy_score", "n_scans"], ascending=False, na_position="last")
    .head(20)[
        ["product_name", "brand", "nutriscore_grade", "nova_group", "controversy_score", "reasons"]
    ]
)
examples.columns = ["Produit", "Marque", "Nutri-Score", "NOVA", "Score", "Explication"]
examples["Nutri-Score"] = examples["Nutri-Score"].str.upper()
examples.style.hide(axis="index").set_properties(subset=["Explication"], **{"width": "420px"})

Produit,Marque,Nutri-Score,NOVA,Score,Explication
Hipro 25g protéines Chocolat,HIPRO,A,4,4,"Nutri-Score A mais ultra-transformé (NOVA 4) ; Nutri-Score A mais 5 additifs ; contient E339 - phosphates de sodium, E452 - Polyphosphates, à risque élevé de surexposition selon l'EFSA ; rappelé le 05/09/2025 (défauts organoleptiques (goût et texture))."
Céleri Rémoulade Pointe de moutarde,Carrefour,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 5 additifs ; contient E202 - Sorbate de potassium, à risque élevé de surexposition selon l'EFSA ; rappelé le 09/05/2025 (corps étranger)."
Tzatziki,Tables du Monde,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 5 additifs ; contient E202 - Sorbate de potassium, à risque élevé de surexposition selon l'EFSA ; rappelé le 19/08/2026 (chaîne du froid)."
"Guacamole (préparation à base d'avocats, de poivrons rouges et depiment)",U,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 6 additifs ; contient E202 - Sorbate de potassium, à risque élevé de surexposition selon l'EFSA ; rappelé le 11/10/2024 (autre contamination microbiologique)."
Galleta sin sal y sin azúcar,CENTROS COMERCIALES CARREFOUR S.A.,A,4,4,"Nutri-Score A mais ultra-transformé (NOVA 4) ; Nutri-Score A mais 5 additifs ; contient E224 - Disulfite de potassium, à risque élevé de surexposition selon l'EFSA ; bio mais ultra-transformé (NOVA 4)."
Tzatziki,Monoprix,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 5 additifs ; contient E202 - Sorbate de potassium, à risque élevé de surexposition selon l'EFSA ; rappelé le 12/01/2022 (oxyde d'éthylène)."
4 Brochettes de dinde nature - Poivrons verts,l'etal du volailler,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 7 additifs ; contient E407 - Carraghénanes, à risque élevé de surexposition selon l'EFSA ; rappelé le 17/05/2023 (listeria)."
Guacamole,L'Apéritif,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 5 additifs ; contient E202 - Sorbate de potassium, à risque élevé de surexposition selon l'EFSA ; rappelé le 12/10/2021 (oxyde d'éthylène)."
1 boîte Cassoulet (2 Pers.),Les Légendaires,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 6 additifs ; contient E250 - Nitrite de sodium, E407a - Algues euchema transformées, E450 - Diphosphates, E451 - Triphosphates, E452 - Polyphosphates, à risque élevé de surexposition selon l'EFSA ; rappelé le 12/08/2021 (pesticides et contaminants chimiques)."
ROTI DE FILET DE POULET CUIT 450g,Volae,B,4,4,"Nutri-Score B mais ultra-transformé (NOVA 4) ; Nutri-Score B mais 6 additifs ; contient E407 - Carraghénanes, à risque élevé de surexposition selon l'EFSA ; rappelé le 03/07/2025 (défaut de cuisson du produit)."


## 8. Synthèse

In [36]:
for line in findings:
    print("-", line)

- 102 012 produits signalés, dont 9 441 avec au moins deux signaux.
- Bien noté mais ultra-transformé : surtout dans le sous-groupe charcuteries (93 % des produits A ou B).
- Additifs à risque élevé : présents dans 32 % des produits NOVA 4.
- Rappels : 6 064 produits d'Open Food Facts retrouvés dans RappelConso.
- Marques : 141 marques ont plus de la moitié de leurs produits A ou B ultra-transformés.
- Bio : 15 947 produits bio ultra-transformés ; parmi les marques bio, jusqu'à 94 % de NOVA 4 (Grain de Sail).


## 9. Limites

- **Score simple, non pondéré** : un rappel sanitaire et un additif à risque de surexposition comptent chacun pour un point, alors qu'ils ne relèvent pas du même registre. Le score sert à repérer et à trier, pas à hiérarchiser des risques.
- **Risque de surexposition ≠ toxicité** : l'EFSA évalue le risque de dépasser la dose journalière admissible à l'échelle de la population, selon les habitudes de consommation. La présence d'un additif dans un produit n'est pas en soi un danger.
- **Données collaboratives** : un produit peut être signalé à tort si sa fiche contient une erreur (liste d'ingrédients obsolète, label mal saisi). Chaque explication renvoie à un fait vérifiable sur la fiche Open Food Facts.
- **Rappels partiels** : RappelConso V2 ne contient que les produits rappelés avec un code-barres, et seuls ceux présents dans Open Food Facts sont retrouvés.
- **Marques** : les classements portent sur les produits documentés dans Open Food Facts, pas sur la gamme complète ni sur les volumes de vente. Ils décrivent des tendances, pas un jugement sur une entreprise.